In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df_events = pd.read_csv("/Users/victoriayuzova/Data-Science-Projects/ba-events-recommender/data/raw/2026-03-16/03_events.csv")
df_events.head()

,url,homepage_url,page_type,title,summary,category,start_date,start_time,venue,price,is_free,tags,confidence,run_date,extracted_at
0,https://complejoteatral.gob.ar/agenda?fecha=16...,https://complejoteatral.gob.ar/,listing,Agenda | Complejo Teatral de Buenos Aires,Listing of theatrical events around 16 March 2...,theatre,NaN,NaN,NaN,NaN,NaN,"['teatro', 'agenda', 'temporada internacional'...",0.80,2026-03-16,2026-03-16T11:43:12+00:00
1,https://complejoteatral.gob.ar/pdf/temporada20...,https://complejoteatral.gob.ar/,pdf,NaN,NaN,other,NaN,NaN,NaN,NaN,NaN,[],0.30,2026-03-16,2026-03-16T11:43:12+00:00
2,https://complejoteatral.gob.ar/ver/GERALD-CLAY...,https://complejoteatral.gob.ar/,event_detail,GERALD CLAYTON TRIO,Comienza el ciclo de jazz en el Teatro San Mar...,music,2026-03-17,20:00,"Teatro San Martín, Sala Martín Coronado","Platea Filas 2 a 5 $85.000, Platea Filas 6 a 2...",False,"['jazz', 'Gerald Clayton', 'Trio', 'Teatro San...",0.95,2026-03-16,2026-03-16T11:43:12+00:00
3,https://complejoteatral.gob.ar/ver/la_gaviota,https://complejoteatral.gob.ar/,event_detail,La Gaviota,"Obra de teatro de Antón Chéjov, dirigida por R...",theatre,NaN,NaN,"Teatro San Martín, Sala Casacuberta","Platea $21.000, Miércoles $12.000",False,"['teatro', 'obra', 'Antón Chéjov', 'Rubén Szuc...",0.95,2026-03-16,2026-03-16T11:43:12+00:00
4,https://complejoteatral.gob.ar/ver/jazz_buenos...,https://complejoteatral.gob.ar/,event_detail,Jazz Buenos Aires,Ciclo de nueve conciertos de Jazz at Lincoln C...,music,2026-03-17,20:00,"Teatro San Martín, Sala Martín Coronado","Platea Filas 2 a 5 $85.000, Platea Filas 6 a 2...",False,"['jazz', 'concierto', 'música', 'Jazz at Linco...",0.95,2026-03-16,2026-03-16T11:43:12+00:00


In [6]:
df_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   url           103 non-null    object 
 1   homepage_url  103 non-null    object 
 2   page_type     103 non-null    object 
 3   title         45 non-null     object 
 4   summary       45 non-null     object 
 5   category      103 non-null    object 
 6   start_date    33 non-null     object 
 7   start_time    33 non-null     object 
 8   venue         44 non-null     object 
 9   price         18 non-null     object 
 10  is_free       22 non-null     object 
 11  tags          103 non-null    object 
 12  confidence    103 non-null    float64
 13  run_date      103 non-null    object 
 14  extracted_at  103 non-null    object 
dtypes: float64(1), object(14)
memory usage: 12.2+ KB


In [7]:
df_clean = df_events[
    (df_events["page_type"]=='event_detail')&
    (df_events["confidence"]>=0.7)&
    (df_events["summary"].notna())
].reset_index(drop=True)

df_clean.shape

(36, 15)

In [8]:
df_clean['category'].value_counts()

category
theatre       9
cinema        7
exhibition    7
talk          7
workshop      3
music         2
dance         1
Name: count, dtype: int64

In [11]:
user_ratings = pd.read_csv('/Users/victoriayuzova/Data-Science-Projects/ba-events-recommender/data/processed/user_ratings.csv')
user_ratings.head()

,url,title,category,is_free,summary,liked
0,https://complejoteatral.gob.ar/ver/GERALD-CLAY...,GERALD CLAYTON TRIO,music,False,Comienza el ciclo de jazz en el Teatro San Mar...,0
1,https://complejoteatral.gob.ar/ver/la_gaviota,La Gaviota,theatre,False,"Obra de teatro de Antón Chéjov, dirigida por R...",1
2,https://complejoteatral.gob.ar/ver/jazz_buenos...,Jazz Buenos Aires,music,False,Ciclo de nueve conciertos de Jazz at Lincoln C...,1
3,https://complejoteatral.gob.ar/ver/los-pilares...,LOS PILARES DE LA SOCIEDAD,theatre,False,Obra de teatro de Henrik Ibsen que denuncia la...,1
4,https://complejoteatral.gob.ar/ver/invasiones-1,INVASIONES I,theatre,False,Invasiones I: No Bombardeen Buenos Aires es un...,0


In [ ]:
merged = df_clean.merge(user_ratings[['url','liked']], on='url', how='left')
print(merged['liked'].notna().sum())

In [16]:
print(merged['category'].value_counts())
print   (f"\nLike rate: {merged['liked'].mean():.2%}")

category
theatre       9
cinema        7
exhibition    7
talk          7
workshop      3
music         2
dance         1
Name: count, dtype: int64

Like rate: 27.78%


In [21]:
merged = merged[['url', 'title', 'summary', 'category', 'venue', 'is_free', 'tags','confidence' ,'homepage_url', 'liked']]

In [22]:
df_clean = merged.copy()
df_clean.to_csv('/Users/victoriayuzova/Data-Science-Projects/ba-events-recommender/data/processed/events_clean.csv', index=False)

print(df_clean.shape)



(36, 10)


In [23]:
df_clean.head()

,url,title,summary,category,venue,is_free,tags,confidence,homepage_url,liked
0,https://complejoteatral.gob.ar/ver/GERALD-CLAY...,GERALD CLAYTON TRIO,Comienza el ciclo de jazz en el Teatro San Mar...,music,"Teatro San Martín, Sala Martín Coronado",False,"['jazz', 'Gerald Clayton', 'Trio', 'Teatro San...",0.95,https://complejoteatral.gob.ar/,0
1,https://complejoteatral.gob.ar/ver/la_gaviota,La Gaviota,"Obra de teatro de Antón Chéjov, dirigida por R...",theatre,"Teatro San Martín, Sala Casacuberta",False,"['teatro', 'obra', 'Antón Chéjov', 'Rubén Szuc...",0.95,https://complejoteatral.gob.ar/,1
2,https://complejoteatral.gob.ar/ver/jazz_buenos...,Jazz Buenos Aires,Ciclo de nueve conciertos de Jazz at Lincoln C...,music,"Teatro San Martín, Sala Martín Coronado",False,"['jazz', 'concierto', 'música', 'Jazz at Linco...",0.95,https://complejoteatral.gob.ar/,1
3,https://complejoteatral.gob.ar/ver/los-pilares...,LOS PILARES DE LA SOCIEDAD,Obra de teatro de Henrik Ibsen que denuncia la...,theatre,"Teatro Presidente Alvear, Av. Corrientes 1659",False,"['Henrik Ibsen', 'teatro', 'obra', 'funciones ...",0.95,https://complejoteatral.gob.ar/,1
4,https://complejoteatral.gob.ar/ver/invasiones-1,INVASIONES I,Invasiones I: No Bombardeen Buenos Aires es un...,theatre,"Teatro San Martín, Sala Martín Coronado",False,"['musical', 'opera rock', 'Elena Roger', 'Char...",0.95,https://complejoteatral.gob.ar/,0


In [ ]:

# Combine summary and tags into one text field
df_clean['text'] = df_clean['summary'] + ' ' + df_clean['tags'].apply(lambda x: ' '.join(x))

# Build TF-IDF matrix
tfidf = TfidfVectorizer(max_features=100)
tfidf_matrix = tfidf.fit_transform(df_clean['text'])

print(tfidf_matrix.shape)

ModuleNotFoundError: No module named 'sklearn'